In [ ]:
from dotenv import load_dotenv
load_dotenv()

from anthropic import Anthropic
import sys
sys.path.append("..")  # helpers.py и prompt_evaluator.py лежат в корне курса
from helpers import add_user_message, chat
from prompt_evaluator import PromptEvaluator

client = Anthropic()
model = "claude-haiku-5-5"
# Без thinking: ответы плана питания не упираются в max_tokens
thinking = {"type": "disabled"}

In [ ]:
# Больше потоков — быстрее, но можно получить ошибку rate limit
evaluator = PromptEvaluator(client, model, max_concurrent_tasks=3)

In [ ]:
dataset = evaluator.generate_dataset(
    task_description="Write a compact, concise 1 day meal plan "
    "for a single athlete",
    prompt_inputs_spec={
        "height": "Athlete's height in cm",
        "weight": "Athlete's weight in kg",
        "goal": "Goal of the athlete",
        "restrictions": "Dietary restrictions of the athlete",
    },
    output_file="15_dataset.json",
    num_cases=3,
)

In [ ]:
def run_prompt(prompt_inputs, system=None):
    """Первая, специально простая версия prompt"""
    prompt = f"""
What should this person eat?

- Height: {prompt_inputs["height"]}
- Weight: {prompt_inputs["weight"]}
- Goal: {prompt_inputs["goal"]}
- Dietary restrictions: {prompt_inputs["restrictions"]}
"""
    messages = []
    add_user_message(messages, prompt)
    return chat(
        client, model, messages, system=system, thinking=thinking
    )

In [ ]:
extra_criteria = """
The output should include:
- Daily caloric total
- Macronutrient breakdown
- Meals with exact foods, portions, and timing
"""

results = evaluator.run_evaluation(
    run_prompt_function=run_prompt,
    dataset_file="15_dataset.json",
    extra_criteria=extra_criteria,
    json_output_file="15_output.json",
    html_output_file="15_output.html",
)

In [ ]:
# Два варианта system prompt для коротких ответов
system_v1 = """
Every word matters. Answer only the question asked. Use simple, common words.
Dry text: no metaphors, no slang, no idioms, adjectives, adverbs,
embellishment or anything else. Add only enough words to make the sentence
understandable.
"""

system_v2 = """
Write short, plain answers. The reader wants facts, not style.
- Use simple, common words.
- Give only what the request asks for. Skip introductions,
  summaries and advice that was not requested.
- Write in short sentences, without adjectives and adverbs
  that add no information.
- Prefer lists and tables to paragraphs.
"""

In [ ]:
results_v1 = evaluator.run_evaluation(
    run_prompt_function=lambda inputs: run_prompt(inputs, system_v1),
    dataset_file="15_dataset.json",
    extra_criteria=extra_criteria,
    json_output_file="15_output_v1.json",
    html_output_file="15_output_v1.html",
)

In [ ]:
results_v2 = evaluator.run_evaluation(
    run_prompt_function=lambda inputs: run_prompt(inputs, system_v2),
    dataset_file="15_dataset.json",
    extra_criteria=extra_criteria,
    json_output_file="15_output_v2.json",
    html_output_file="15_output_v2.html",
)

In [ ]:
# Требования из extra_criteria теперь прямо в prompt
def run_prompt_with_criteria(prompt_inputs, system=None):
    prompt = f"""
What should this person eat?

- Height: {prompt_inputs["height"]}
- Weight: {prompt_inputs["weight"]}
- Goal: {prompt_inputs["goal"]}
- Dietary restrictions: {prompt_inputs["restrictions"]}

The output should include:
- Daily caloric total
- Macronutrient breakdown
- Meals with exact foods, portions, and timing
"""
    messages = []
    add_user_message(messages, prompt)
    return chat(
        client, model, messages, system=system, thinking=thinking
    )

In [ ]:
results_v3 = evaluator.run_evaluation(
    run_prompt_function=lambda inputs: run_prompt_with_criteria(
        inputs, system_v2
    ),
    dataset_file="15_dataset.json",
    extra_criteria=extra_criteria,
    json_output_file="15_output_v3.json",
    html_output_file="15_output_v3.html",
)